# Phase 7: GPS 2D Pre-training (300k, Kaggle T4)
Graph Transformer on 2D bond-topology graphs.
Outputs: `gps_2d_encoder.pt` + `gps_2d_best.pt` + embeddings

In [ ]:
!pip install -q torch-geometric optuna

In [ ]:
import torch, os, json, time
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
import torch.multiprocessing as mp
from torch_geometric.loader import DataLoader, DataListLoader
from torch_geometric.nn import GPSConv, GINEConv, global_mean_pool
from torch_geometric.nn import DataParallel as PyGDataParallel
import optuna

# Performance: enable cudnn autotuner.
torch.backends.cudnn.benchmark = True

N_GPUS = torch.cuda.device_count()
USE_MULTI_GPU = N_GPUS > 1
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

print(f"torch {torch.__version__}, CUDA {torch.cuda.is_available()}")
for i in range(N_GPUS):
    props = torch.cuda.get_device_properties(i)
    print(f"GPU {i}: {props.name}, {props.total_memory / 1e9:.1f} GB")
print(f"Mode: {'PyG DataParallel (' + str(N_GPUS) + ' GPUs)' if USE_MULTI_GPU else 'single GPU'}")

# DataLoader settings (Kaggle has ~4 CPU cores).
# NOTE on multi-GPU: PyG DataParallel needs DataListLoader, which hands workers
# a *list of individual Data objects* (not a collated Batch). Each small tensor
# is then shared cross-process via mmap; at large batch this exhausts Kaggle's
# /dev/shm and crashes with "unable to mmap ... Cannot allocate memory (12)".
# Neither sharing strategy (file_system nor file_descriptor) avoids it. So the
# multi-GPU path MUST use workers=0 (main-thread collation). Single-GPU is fine
# with real workers because DataLoader collates before sharing.
NUM_WORKERS = 2
PIN_MEMORY = True
LOADER_NUM_WORKERS = 0 if USE_MULTI_GPU else NUM_WORKERS
LOADER_PIN_MEMORY = PIN_MEMORY and not USE_MULTI_GPU

SAVE_DIR = '/kaggle/working'

# Auto-detect input path
for candidate in [
    '/kaggle/input/pyg-2d-graphs-bond-300k',
    '/kaggle/input/datasets/nothingnessvoid/pyg-2d-graphs-bond-300k',
]:
    if os.path.isdir(candidate):
        INPUT_DIR = candidate
        break
else:
    raise FileNotFoundError('Dataset not found')

print('Input:', INPUT_DIR)
print('Save:', SAVE_DIR)
print(f'Loader workers: {LOADER_NUM_WORKERS}, pin_memory={LOADER_PIN_MEMORY}')

In [ ]:
# Load 2D graphs
graphs = torch.load(f'{INPUT_DIR}/pyg_2d_graphs_bond_300k.pt', weights_only=False)
print(f'Loaded {len(graphs)} 2D graphs')
print(f'Node features: {graphs[0].x.shape[1]}, Edge features: {graphs[0].edge_attr.shape[1]}')

SEED = 42
N = len(graphs)
idx = np.random.RandomState(SEED).permutation(N)

# Full splits (for retrain + test + embeddings)
n_train = int(0.8 * N)
n_val = int(0.1 * N)
train_set_full = [graphs[i] for i in idx[:n_train]]
val_set_full = [graphs[i] for i in idx[n_train:n_train+n_val]]
test_set = [graphs[i] for i in idx[n_train+n_val:]]

# Small splits for Optuna (50k total -> 40k train, 5k val)
OPTUNA_LIMIT = 50000
idx_small = np.random.RandomState(SEED).permutation(OPTUNA_LIMIT)
n_train_s = int(0.8 * OPTUNA_LIMIT)
n_val_s = int(0.1 * OPTUNA_LIMIT)
train_set_small = [graphs[i] for i in idx_small[:n_train_s]]
val_set_small = [graphs[i] for i in idx_small[n_train_s:n_train_s+n_val_s]]

print(f'Full split: train={len(train_set_full)}, val={len(val_set_full)}, test={len(test_set)}')
print(f'Optuna split: train={len(train_set_small)}, val={len(val_set_small)}')

In [ ]:
class GPSWrapper(nn.Module):
    def __init__(self, in_channels=9, edge_dim=4, hidden_channels=128,
                 num_layers=6, num_heads=8, dropout=0.1, n_targets=3):
        super().__init__()
        self.node_emb = nn.Linear(in_channels, hidden_channels)
        self.edge_emb = nn.Linear(edge_dim, hidden_channels)
        self.convs = nn.ModuleList()
        for _ in range(num_layers):
            gin = GINEConv(
                nn.Sequential(
                    nn.Linear(hidden_channels, hidden_channels),
                    nn.SiLU(),
                    nn.Linear(hidden_channels, hidden_channels),
                ), edge_dim=hidden_channels)
            gps = GPSConv(channels=hidden_channels, conv=gin,
                          heads=num_heads, dropout=dropout,
                          act='silu', norm='batch_norm',
                          attn_type='multihead')
            self.convs.append(gps)
        self.hidden_channels = hidden_channels
        self.head = nn.Sequential(
            nn.Linear(hidden_channels, hidden_channels),
            nn.SiLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_channels, hidden_channels // 2),
            nn.SiLU(),
            nn.Linear(hidden_channels // 2, n_targets),
        )

    def encode(self, x, edge_index, edge_attr, batch):
        h = self.node_emb(x.float())
        e = self.edge_emb(edge_attr.float())
        for conv in self.convs:
            h = conv(h, edge_index, batch, edge_attr=e)
        return global_mean_pool(h, batch)

    def forward(self, data_or_x, edge_index=None, edge_attr=None, batch=None,
                return_embeddings=False):
        # PyG DataParallel calls the replica as model(batch_obj) with a single
        # Data object; the standalone/eval path calls model(x, edge_index, ...).
        if edge_index is None and hasattr(data_or_x, 'x'):
            data = data_or_x
            h = self.encode(data.x, data.edge_index, data.edge_attr, data.batch)
        else:
            h = self.encode(data_or_x, edge_index, edge_attr, batch)
        return h if return_embeddings else self.head(h)

print('GPSWrapper defined')

In [ ]:
# ── Multi-GPU-aware training helpers ──
def make_loader(dataset, batch_size, shuffle):
    """DataListLoader for DataParallel (lists of Data, scattered per-GPU);
    plain DataLoader (collated Batch) for single GPU. Both use real workers:
    DataParallel's main-thread scatter otherwise starves the GPUs (~55% util).
    file_system sharing (set above) keeps workers safe on Kaggle's small shm."""
    if USE_MULTI_GPU:
        return DataListLoader(dataset, batch_size=batch_size, shuffle=shuffle,
                              num_workers=LOADER_NUM_WORKERS, pin_memory=False,
                              persistent_workers=LOADER_NUM_WORKERS > 0)
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle,
                      num_workers=LOADER_NUM_WORKERS, pin_memory=LOADER_PIN_MEMORY,
                      persistent_workers=LOADER_NUM_WORKERS > 0)


def _state_dict(model):
    """Strip the DataParallel `module.` prefix so checkpoints load into a
    plain GPSWrapper locally."""
    return model.module.state_dict() if USE_MULTI_GPU else model.state_dict()


def _prepare_batch(batch):
    """Return (model_input, y_on_device). Multi-GPU keeps the list (PyG
    DataParallel scatters it); single-GPU collates to a device Batch."""
    if USE_MULTI_GPU and isinstance(batch, list):
        y = torch.cat([d.y for d in batch], dim=0).to(device)
        return batch, y
    batch = batch.to(device)
    return batch, batch.y


def _forward(model, batch):
    if USE_MULTI_GPU and isinstance(batch, list):
        return model(batch)  # PyG DataParallel scatters the Data list
    return model(batch.x, batch.edge_index, batch.edge_attr, batch.batch)


def run_training(params, train_set, val_set, max_epochs=80,
                 patience=15, resume_ckpt=None, save_prefix='gps',
                 verbose=True, log_every=1, phase_label=''):
    base_model = GPSWrapper(
        hidden_channels=params['hidden_channels'],
        num_layers=params['num_layers'],
        num_heads=params['num_heads'],
        dropout=params['dropout'],
    )
    model = PyGDataParallel(base_model).to(device) if USE_MULTI_GPU else base_model.to(device)

    optimizer = torch.optim.AdamW(model.parameters(),
                                  lr=params['lr'],
                                  weight_decay=params['weight_decay'])
    if params['scheduler'] == 'plateau':
        sched = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, patience=5, factor=0.5, min_lr=1e-6)
    else:
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=max_epochs, eta_min=1e-6)

    scaler = torch.amp.GradScaler('cuda')
    criterion = nn.L1Loss()

    start_epoch = 0
    if resume_ckpt and os.path.exists(resume_ckpt):
        ckpt = torch.load(resume_ckpt, weights_only=False)
        # checkpoints store the unwrapped state_dict
        (model.module if USE_MULTI_GPU else model).load_state_dict(ckpt['model'])
        optimizer.load_state_dict(ckpt['optimizer'])
        sched.load_state_dict(ckpt['scheduler'])
        scaler.load_state_dict(ckpt['scaler'])
        start_epoch = ckpt['epoch'] + 1
        print(f'Resumed from epoch {start_epoch}')

    bs = params['batch_size']
    train_loader = make_loader(train_set, bs, True)
    val_loader = make_loader(val_set, bs, False)
    tag = f"[{phase_label}] " if phase_label else ""
    if verbose:
        print(f"  {tag}loaders: workers={train_loader.num_workers}, "
              f"use_data_list={USE_MULTI_GPU}, bs={bs}", flush=True)

    best_val = float('inf')
    wait = 0

    for epoch in range(start_epoch, max_epochs):
        model.train()
        t0 = time.time()
        train_loss = 0.0
        n_seen = 0
        for batch_data in train_loader:
            inp, y = _prepare_batch(batch_data)
            optimizer.zero_grad()
            with torch.amp.autocast('cuda'):
                pred = _forward(model, inp)
                loss = criterion(pred, y)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 10.0)
            scaler.step(optimizer)
            scaler.update()
            train_loss += loss.item() * y.size(0)
            n_seen += y.size(0)
        train_loss /= max(n_seen, 1)

        model.eval()
        val_loss = 0.0
        n_val = 0
        with torch.no_grad():
            for batch_data in val_loader:
                inp, y = _prepare_batch(batch_data)
                with torch.amp.autocast('cuda'):
                    pred = _forward(model, inp)
                    loss = criterion(pred, y)
                val_loss += loss.item() * y.size(0)
                n_val += y.size(0)
        val_loss /= max(n_val, 1)

        if params['scheduler'] == 'plateau':
            sched.step(val_loss)
        else:
            sched.step()

        elapsed = time.time() - t0
        lr_now = optimizer.param_groups[0]['lr']

        is_best = val_loss < best_val
        if is_best:
            best_val = val_loss
            wait = 0
            torch.save(_state_dict(model), f'{SAVE_DIR}/{save_prefix}_best.pt')
        else:
            wait += 1

        if verbose and (epoch % log_every == 0 or epoch == start_epoch or is_best):
            marker = ' *' if is_best else ''
            print(f'  {tag}ep{epoch:03d}/{max_epochs} train={train_loss:.4f} '
                  f'val={val_loss:.4f} best={best_val:.4f} lr={lr_now:.2e} '
                  f'{elapsed:.0f}s{marker}', flush=True)

        if epoch % 5 == 0:
            torch.save({
                'epoch': epoch, 'model': _state_dict(model),
                'optimizer': optimizer.state_dict(),
                'scheduler': sched.state_dict(),
                'scaler': scaler.state_dict(),
                'best_val': best_val,
            }, f'{SAVE_DIR}/{save_prefix}_ckpt.pt')

        if wait >= patience:
            if verbose:
                print(f'  {tag}Early stop at epoch {epoch}', flush=True)
            break

    return best_val

print('run_training defined')

In [ ]:
# Best params from previous Optuna run (20 trials on 50k subset)
# Skip Optuna search — use hardcoded results directly
best_params = {
    'hidden_channels': 192,
    'num_layers': 7,
    'num_heads': 4,
    'dropout': 0.05,
    'lr': 0.0004754654349367296,
    'weight_decay': 1.3094136884618282e-05,
    'batch_size': 256,
    'scheduler': 'cosine',
}
print('Using saved best params (Optuna search skipped):')
print(f'  MAE=0.1445 (on 50k subset)')
print(best_params)

In [ ]:
# Retrain best on FULL 300k data
bp = dict(best_params)

# GPS attention is compute-dense, so it actually fills both GPUs. On T4 x2 the
# per-GPU share is bp['batch_size'] // 2, so scale the global batch up to keep
# each card busy. Linear-scaling rule: when batch grows k x, grow lr ~k x too.
# Set MULTI_GPU_BATCH_SCALE = 1 to keep the single-GPU-tuned config unchanged.
MULTI_GPU_BATCH_SCALE = 2
if USE_MULTI_GPU and MULTI_GPU_BATCH_SCALE > 1:
    bp['batch_size'] = bp['batch_size'] * MULTI_GPU_BATCH_SCALE
    bp['lr'] = bp['lr'] * MULTI_GPU_BATCH_SCALE
    print(f"Multi-GPU: scaled batch_size -> {bp['batch_size']}, lr -> {bp['lr']:.2e} "
          f"(x{MULTI_GPU_BATCH_SCALE} linear scaling)")

print('\nRetraining best config on full 300k...')
best_val = run_training(bp, train_set_full, val_set_full,
                        max_epochs=150, patience=25,
                        save_prefix='gps_2d_best',
                        verbose=True, log_every=1, phase_label='retrain')
print(f'\nRetrain done. Best val MAE: {best_val:.4f}')

In [ ]:
# Test evaluation
bp = best_params
model = GPSWrapper(
    hidden_channels=bp['hidden_channels'],
    num_layers=bp['num_layers'],
    num_heads=bp['num_heads'],
    dropout=bp['dropout'],
).to(device)
model.load_state_dict(torch.load(f'{SAVE_DIR}/gps_2d_best_best.pt',
                                  weights_only=False))
model.eval()

test_loader = DataLoader(test_set, batch_size=256, shuffle=False, num_workers=2)
from sklearn.metrics import mean_absolute_error, r2_score

all_pred, all_true = [], []
with torch.no_grad():
    for batch_data in test_loader:
        batch_data = batch_data.to(device)
        pred = model(batch_data.x, batch_data.edge_index,
                     batch_data.edge_attr, batch_data.batch)
        all_pred.append(pred.cpu().numpy())
        all_true.append(batch_data.y.cpu().numpy())

all_pred = np.concatenate(all_pred)
all_true = np.concatenate(all_true)

labels = ['HOMO', 'LUMO', 'Gap']
metrics = {}
for i, name in enumerate(labels):
    mae = mean_absolute_error(all_true[:, i], all_pred[:, i])
    r2 = r2_score(all_true[:, i], all_pred[:, i])
    print(f'{name}: MAE={mae:.4f} eV, R2={r2:.4f}')
    metrics[name] = {'mae': float(mae), 'r2': float(r2)}

metrics['best_params'] = bp
with open(f'{SAVE_DIR}/gps_2d_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print('Metrics saved')

In [ ]:
# Save model weights + extract embeddings for fusion
print('Saving model weights...')
torch.save(model.state_dict(), f'{SAVE_DIR}/gps_2d_300k.pt')
print(f'  Saved gps_2d_300k.pt')

print('Extracting 2D embeddings...')
all_loader = DataLoader(graphs, batch_size=256, shuffle=False, num_workers=2)
embeddings_2d = []
with torch.no_grad():
    for batch_data in all_loader:
        batch_data = batch_data.to(device)
        emb = model.encode(batch_data.x, batch_data.edge_index,
                           batch_data.edge_attr, batch_data.batch)
        embeddings_2d.append(emb.cpu())
embeddings_2d = torch.cat(embeddings_2d, dim=0)
print(f'2D embeddings: {embeddings_2d.shape}')
torch.save(embeddings_2d, f'{SAVE_DIR}/gps_2d_embeddings.pt')

print('\nOutput files:')
for f in os.listdir(SAVE_DIR):
    size = os.path.getsize(f'{SAVE_DIR}/{f}') / 1e6
    print(f'  {f} ({size:.1f} MB)')
print('Done!')